In [ ]:
# imports and paths

from pathlib import Path
import re

import numpy as np
import pandas as pd

RAW_SCADA_DIR = Path("Penmanshiel")
STATIC_CSV_PATH = RAW_SCADA_DIR / "Penmanshiel_WT_static.csv"

PROCESSED_DIR = Path("processed_data")
PROCESSED_DIR.mkdir(exist_ok=True)

RAW_SCADA_OUTPUT = PROCESSED_DIR / "penmanshiel_scada_raw.parquet"
CLEAN_SCADA_OUTPUT = PROCESSED_DIR / "penmanshiel_scada_clean.parquet"
COMPLETE_SCADA_OUTPUT = PROCESSED_DIR / "penmanshiel_scada_complete.parquet"
STATIC_OUTPUT = PROCESSED_DIR / "penmanshiel_static.parquet"
QUALITY_OUTPUT = PROCESSED_DIR / "penmanshiel_data_quality_summary.csv"

ANALYSIS_END = pd.Timestamp("2023-01-01", tz="UTC")
N_TURBINES = 14
TURBINE_RATED_POWER_KW = 2050


In [ ]:
# columns retained from the raw scada files

RAW_COLUMNS = [
    "timestamp",
    "Wind speed (m/s)",
    "Density adjusted wind speed (m/s)",
    "Wind direction (°)",
    "Nacelle position (°)",
    "Power (kW)",
    "Potential power default PC (kW)",
    "Lost Production to Curtailment (Total) (kWh)",
    "Rotor speed (RPM)",
    "Blade angle (pitch position) A (°)",
    "Blade angle (pitch position) B (°)",
    "Blade angle (pitch position) C (°)",
    "Turbine Power setpoint (kW)",
    "Turbine Power setpoint, Max (kW)",
    "Turbine Power setpoint, Min (kW)",
    "Turbine Power setpoint, StdDev (kW)",
    "Available Capacity for Production (kW)",
    "Data Availability",
    "Lost Production to Downtime (kWh)",
]

COLUMN_RENAME = {
    "Wind speed (m/s)": "wind_speed",
    "Density adjusted wind speed (m/s)": "wind_speed_density",
    "Wind direction (°)": "wind_dir",
    "Nacelle position (°)": "nacelle_pos",
    "Power (kW)": "power_kw",
    "Potential power default PC (kW)": "potential_power_kw",
    "Lost Production to Curtailment (Total) (kWh)": "curtailment_kwh",
    "Rotor speed (RPM)": "rotor_rpm",
    "Blade angle (pitch position) A (°)": "pitch_a",
    "Blade angle (pitch position) B (°)": "pitch_b",
    "Blade angle (pitch position) C (°)": "pitch_c",
    "Turbine Power setpoint (kW)": "power_setpoint_kw",
    "Turbine Power setpoint, Max (kW)": "power_setpoint_max_kw",
    "Turbine Power setpoint, Min (kW)": "power_setpoint_min_kw",
    "Turbine Power setpoint, StdDev (kW)": "power_setpoint_std_kw",
    "Available Capacity for Production (kW)": "available_capacity_kw",
    "Data Availability": "data_availability",
    "Lost Production to Downtime (kWh)": "downtime_kwh",
}


In [ ]:
# load one turbine scada file

def load_turbine_file(file_path):
    turbine_match = re.search(r"Penmanshiel_(\d+)", file_path.name)

    if turbine_match is None:
        raise ValueError(f"Could not identify turbine number from {file_path.name}")

    turbine_id = int(turbine_match.group(1))

    data = pd.read_csv(
        file_path,
        skiprows=9,
        encoding="utf-8-sig",
        low_memory=False,
    )

    data = data.rename(columns={"# Date and time": "timestamp"})

    missing_columns = [
        column for column in RAW_COLUMNS
        if column not in data.columns
    ]

    if missing_columns:
        raise ValueError(
            f"{file_path.name} is missing required columns: {missing_columns}"
        )

    data = data[RAW_COLUMNS].copy()
    data = data.rename(columns=COLUMN_RENAME)

    data["timestamp"] = pd.to_datetime(
        data["timestamp"],
        utc=True,
        errors="coerce",
    )

    data["turbine_id"] = turbine_id

    return data


In [ ]:
# discover and load files used in the dissertation period

all_scada_files = sorted(
    RAW_SCADA_DIR.rglob("Turbine_Data_Penmanshiel_*.csv")
)

if not all_scada_files:
    raise FileNotFoundError(
        f"No Penmanshiel SCADA files found under {RAW_SCADA_DIR.resolve()}"
    )

loaded_files = []

for file_number, file_path in enumerate(all_scada_files, start=1):
    turbine_data = load_turbine_file(file_path)
    loaded_files.append(turbine_data)

scada_raw = pd.concat(
    loaded_files,
    ignore_index=True,
)

scada_raw = scada_raw[
    scada_raw["timestamp"] < ANALYSIS_END
].copy()

scada_raw = (
    scada_raw
    .sort_values(["turbine_id", "timestamp"])
    .reset_index(drop=True)
)

print(f"rows loaded: {len(scada_raw):,}")
print(f"turbines: {scada_raw['turbine_id'].nunique()}")
print(f"period: {scada_raw['timestamp'].min()} to {scada_raw['timestamp'].max()}")


In [ ]:
# summarise the data-quality issues reported in chapter 3

total_rows = len(scada_raw)

quality_counts = {
    "Missing power": scada_raw["power_kw"].isna().sum(),
    "Missing wind speed": scada_raw["wind_speed"].isna().sum(),
    "Negative power": (scada_raw["power_kw"] < 0).sum(),
    "Power above 2050 kW": (scada_raw["power_kw"] > TURBINE_RATED_POWER_KW).sum(),
    "Wind speed at or below 0 m/s": (scada_raw["wind_speed"] <= 0).sum(),
    "Rotor speed at or below 0 rpm": (scada_raw["rotor_rpm"] <= 0).sum(),
}

quality_summary = pd.DataFrame(
    {
        "data_quality_issue": list(quality_counts.keys()),
        "observations": list(quality_counts.values()),
    }
)

quality_summary["raw_data_pct"] = (
    quality_summary["observations"] / total_rows * 100
)

duplicate_count = scada_raw.duplicated(
    subset=["turbine_id", "timestamp"]
).sum()

print(quality_summary.to_string(index=False))
print(f"\nduplicate turbine-timestamps: {duplicate_count:,}")


In [ ]:
# apply the turbine-level quality filters

required_clean_columns = [
    "timestamp",
    "turbine_id",
    "wind_speed",
    "wind_dir",
    "power_kw",
    "rotor_rpm",
]

scada_clean = scada_raw.dropna(
    subset=required_clean_columns
).copy()

scada_clean = scada_clean[
    scada_clean["power_kw"].between(
        0,
        TURBINE_RATED_POWER_KW,
        inclusive="both",
    )
    & (scada_clean["wind_speed"] > 0)
    & (scada_clean["rotor_rpm"] > 0)
].copy()

scada_clean = (
    scada_clean
    .sort_values(["timestamp", "turbine_id"])
    .reset_index(drop=True)
)

print(f"clean turbine observations: {len(scada_clean):,}")


In [ ]:
# retain timestamps with valid observations from all 14 turbines

turbines_per_timestamp = (
    scada_clean
    .groupby("timestamp")["turbine_id"]
    .nunique()
)

complete_timestamps = turbines_per_timestamp[
    turbines_per_timestamp == N_TURBINES
].index

scada_complete = scada_clean[
    scada_clean["timestamp"].isin(complete_timestamps)
].copy()

scada_complete = (
    scada_complete
    .sort_values(["timestamp", "turbine_id"])
    .reset_index(drop=True)
)

print(f"complete farm timestamps: {len(complete_timestamps):,}")
print(f"complete turbine observations: {len(scada_complete):,}")
print(f"period: {complete_timestamps.min()} to {complete_timestamps.max()}")


In [ ]:
# prepare the static turbine data and local metre coordinates

static = pd.read_csv(STATIC_CSV_PATH)

static = static.dropna(
    subset=["Latitude"]
).copy()

static["turbine_id"] = (
    static["Alternative Title"]
    .str.replace("T", "", regex=False)
    .astype(int)
)

static = static[
    [
        "turbine_id",
        "Latitude",
        "Longitude",
        "Elevation (m)",
        "Hub Height (m)",
        "Rotor Diameter (m)",
        "Rated power (kW)",
    ]
].rename(
    columns={
        "Latitude": "lat",
        "Longitude": "lon",
        "Elevation (m)": "elevation_m",
        "Hub Height (m)": "hub_height_m",
        "Rotor Diameter (m)": "rotor_diameter_m",
        "Rated power (kW)": "rated_power_kw",
    }
)

farm_centre_lat = static["lat"].mean()
farm_centre_lon = static["lon"].mean()

metres_per_degree_lat = 111_320
metres_per_degree_lon = (
    111_320 * np.cos(np.deg2rad(farm_centre_lat))
)

static["x_m"] = (
    static["lon"] - farm_centre_lon
) * metres_per_degree_lon

static["y_m"] = (
    static["lat"] - farm_centre_lat
) * metres_per_degree_lat

static = static.sort_values("turbine_id").reset_index(drop=True)

print(static.to_string(index=False))


In [ ]:
# save the processed datasets used by the remaining notebooks

scada_raw.to_parquet(
    RAW_SCADA_OUTPUT,
    index=False,
)

scada_clean.to_parquet(
    CLEAN_SCADA_OUTPUT,
    index=False,
)

scada_complete.to_parquet(
    COMPLETE_SCADA_OUTPUT,
    index=False,
)

static.to_parquet(
    STATIC_OUTPUT,
    index=False,
)

quality_summary.to_csv(
    QUALITY_OUTPUT,
    index=False,
)

print(f"saved {RAW_SCADA_OUTPUT}")
print(f"saved {CLEAN_SCADA_OUTPUT}")
print(f"saved {COMPLETE_SCADA_OUTPUT}")
print(f"saved {STATIC_OUTPUT}")
print(f"saved {QUALITY_OUTPUT}")
